# 02 — Tiền xử lý xLAM về định dạng hàng huấn luyện (parity) và chia train/val/test

**Nhiệm vụ ClickUp:** Tuần 2 — Tiền xử lý về định dạng hàng huấn luyện (parity, `<tool_call>`) và chia train/val/test

**Mục tiêu:** chạy `datagen/convert_xlam.py` (stdlib, vài giây, CPU) để tự tái tạo bộ chia 1600/200/200 từ
`data/public/xlam_raw_2k.jsonl`, chứng minh kết quả **trùng bit-for-bit** với bản đã commit, kiểm tra 0 tool-set dùng chung
giữa các split, chạy cổng kiểm định `training/validate_dataset.py` (exit 0), xem **một** prompt đã render có khối `<tools>`,
và đo độ dài token (con số `% > 2560` — `--max-len` của F Tuần 1 — được nhóm Fine-tune dùng). Đây là đầu vào của **Điểm đồng bộ 1**.

Cách chạy: `Runtime → Run all`. Mọi file tạm ghi ra `/tmp/` — notebook **không** ghi đè `data/public/`.

In [ ]:
# --- Thiết lập (Colab + local) --------------------------------------------------------------
# Colab : read GITHUB_TOKEN from Secrets, clone the private repo (skip if present), chdir into it.
# Local : walk up from the current directory until the repo root (docs/contracts/cli.md) is found.
# Sets REPO (Path), GIT_SHA, IN_COLAB, AUTHOR and a run() helper that calls repo scripts.
import os, shlex, subprocess, sys
from pathlib import Path

REPO_HTTPS = "github.com/thanhhao98/ChatSystem"
MARKER = "docs/contracts/cli.md"          # exists at the root of every checkout

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


def _github_token():
    # Colab Secrets -> Kaggle Secrets -> environment variable. Never print the value.
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    return os.environ["GITHUB_TOKEN"]


if IN_COLAB or IN_KAGGLE:
    try:
        _token = _github_token()
    except Exception as e:  # SecretNotFoundError / NotebookAccessError / KeyError
        raise RuntimeError(
            "Thiếu secret GITHUB_TOKEN. Colab: biểu tượng chìa khoá (Secrets) -> Add new secret: "
            "Name = GITHUB_TOKEN, Value = Personal access token (classic, scope repo) của tài khoản collaborator "
            "trên thanhhao98/ChatSystem, bật 'Notebook access'. Kaggle: Add-ons -> Secrets -> GITHUB_TOKEN. "
            "Rồi chạy lại ô này.") from e
    if not Path("ChatSystem").exists():
        _r = subprocess.run(["git", "clone", "--quiet", f"https://{_token}@{REPO_HTTPS}", "ChatSystem"],
                            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("git clone thất bại: " + _r.stderr.replace(_token, "<token>"))
    os.chdir("ChatSystem")
    del _token
else:
    _here = Path.cwd().resolve()
    for _cand in [_here, *_here.parents]:
        if (_cand / MARKER).exists():
            os.chdir(_cand)
            break
    else:
        raise FileNotFoundError(f"Không tìm thấy gốc repo (không có {MARKER}) khi đi lên từ {_here}. "
                                "Mở notebook từ bên trong thư mục ChatSystem đã clone.")

REPO = Path.cwd()


def _git(*args):
    # Small helper: run a git command in REPO and return stdout ("" on any failure).
    try:
        return subprocess.run(["git", *args], cwd=REPO, capture_output=True, text=True).stdout.strip()
    except OSError:
        return ""


GIT_SHA = _git("rev-parse", "--short", "HEAD") or "no-git"
AUTHOR = os.environ.get("GITHUB_USER") or _git("config", "user.name") or "điền tên"


def run(cmd):
    # Run a repo script (list of args), echo the command, stream its output, return CompletedProcess.
    shown = " ".join(shlex.quote(c) for c in cmd).replace(shlex.quote(sys.executable), "python", 1)
    print("$ " + shown)
    p = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout.rstrip())
    if p.stderr:
        print(p.stderr.rstrip())
    print(f"[exit code = {p.returncode}]")
    return p


print(f"REPO      = {REPO}")
print(f"git HEAD  = {GIT_SHA}")
print(f"python    = {sys.version.split()[0]} · Colab = {IN_COLAB} · Kaggle = {IN_KAGGLE} · author = {AUTHOR}")

## Định dạng hàng huấn luyện "parity" trong 5 điểm

Hợp đồng đầy đủ: [`docs/contracts/training_row_format.md`](../../docs/contracts/training_row_format.md)
(đóng băng tại Điểm đồng bộ 1, 2026-09-27). Tóm tắt:

1. **Một hàng = một JSON** `{"id", "role", "messages": [system, user, assistant], "replay": true, "replay_tools": [...]}` —
   đúng **3** message, không hơn không kém (phase 2 chỉ dùng đơn lượt).
2. **`messages[0].content` = nguyên văn preamble** trong `prompts/system_preamble_v0.txt` (so sánh sau `.strip()`).
   Hàng công khai dùng v0 (trung lập, không có danh từ nghiệp vụ); hàng SGOD sau này dùng v1.
3. **Tool không bao giờ nằm trong text của message.** Danh sách tool ở `replay_tools` (dạng OpenAI
   `{"type": "function", "function": {name, description, parameters}}`) và được template chat của Qwen render thành khối
   `<tools>…</tools>` **lúc train lẫn lúc serve** qua `tokenizer.apply_chat_template(..., tools=...)` → train/serve giống nhau
   từng ký tự (bài học +18pp ở hệ thống tham chiếu (POC v1)).
4. **Assistant target** là một hoặc nhiều `<tool_call>{"name": …, "arguments": {…}}</tool_call>` **hoặc** đúng chuỗi
   `REFUSAL_VI` / `DEFLECT_VI` trong `prompts/fixed_replies.json` — không có dạng thứ ba.
5. **Chia tập theo tool-set**: mọi hàng có cùng tập tool đi vào cùng một split → 0 tool-set dùng chung giữa train/val/test.
   `MAX_SEQ_LENGTH` = 4096; hàng dài hơn `--max-len` khi train là **lỗi cứng**, không cắt ngầm. Cổng kiểm tra máy:
   `training/validate_dataset.py` (cờ trong `docs/contracts/cli.md`).

## Bước 1 — Chạy bộ chuyển đổi ra `/tmp`

In [ ]:
# Re-run the converter into /tmp with the recorded seed (never overwrite data/public/ from a notebook).
import json
from pathlib import Path

OUT_PREFIX = "/tmp/xlam_2k"
SEED = 20260913
p_convert = run([sys.executable, "datagen/convert_xlam.py",
                 "--in", "data/public/xlam_raw_2k.jsonl",
                 "--out-prefix", OUT_PREFIX,
                 "--system-file", "prompts/system_preamble_v0.txt",
                 "--seed", str(SEED)])
assert p_convert.returncode == 0, "convert_xlam.py thất bại — đọc thông báo lỗi ở trên"

## Bước 2 — So khớp với bản đã commit (sha256 danh sách id)

In [ ]:
# sha256 over the newline-joined id list, same recipe as convert_xlam.py; plus a raw byte comparison.
import hashlib


def read_jsonl(path):
    return [json.loads(l) for l in Path(path).open(encoding="utf-8") if l.strip()]


def ids_sha256(rows):
    return hashlib.sha256("\n".join(r["id"] for r in rows).encode()).hexdigest()


SPLITS = {}
for split in ["train", "val", "test"]:
    new_path, old_path = Path(f"{OUT_PREFIX}.{split}.jsonl"), REPO / f"data/public/xlam_2k.{split}.jsonl"
    new, old = read_jsonl(new_path), read_jsonl(old_path)
    sha_new, sha_old = ids_sha256(new), ids_sha256(old)
    SPLITS[split] = {"rows": len(new), "rows_committed": len(old), "ids_sha256": sha_new,
                     "match": sha_new == sha_old, "bytes_match": new_path.read_bytes() == old_path.read_bytes()}
    print(f"{split:5s} rows={len(new):4d} (committed {len(old):4d}) ids_sha256={sha_new[:16]}  "
          f"split matches committed: {SPLITS[split]['match']}  (bytes identical: {SPLITS[split]['bytes_match']})")

# eval.json is derived from the test split -> its id order must match too
eval_new = json.load(open(f"{OUT_PREFIX}.eval.json", encoding="utf-8"))
eval_old = json.load(open(REPO / "data/public/xlam_2k.eval.json", encoding="utf-8"))
EVAL_MATCH = [e["id"] for e in eval_new] == [e["id"] for e in eval_old]
print(f"eval  records={len(eval_new)} ids match committed: {EVAL_MATCH}")
assert all(s["match"] for s in SPLITS.values()) and EVAL_MATCH, "Bộ chia KHÔNG trùng bản commit — kiểm tra seed / preamble / file raw"

## Bước 3 — Kiểm tra rò rỉ: tool-set dùng chung giữa các split phải là 0

In [ ]:
# A tool-set (sorted tuple of tool names) must live in exactly one split.
def tool_set(row):
    return tuple(sorted(t["function"]["name"] for t in row["replay_tools"]))


sets = {s: {tool_set(r) for r in read_jsonl(f"{OUT_PREFIX}.{s}.jsonl")} for s in SPLITS}
for s, ts in sets.items():
    SPLITS[s]["tool_sets"] = len(ts)
    print(f"{s:5s} distinct tool-sets = {len(ts)}")
SHARED = (sets["train"] & sets["val"]) | (sets["train"] & sets["test"]) | (sets["val"] & sets["test"])
print(f"tool-sets shared across splits = {len(SHARED)}  (phải là 0)")
assert len(SHARED) == 0, f"Rò rỉ: {sorted(SHARED)[:5]}"

## Bước 4 — Cổng kiểm định `training/validate_dataset.py`

Exit code **0** = mọi hàng hợp lệ theo hợp đồng 1. Hàng công khai có `replay_tools` nên **không** cần `--tools`.

In [ ]:
# Machine gate for contract 1 (flags: docs/contracts/cli.md). Exit code 0 = every row valid.
VALIDATOR = REPO / "training/validate_dataset.py"
VALIDATE_EXIT = None
if VALIDATOR.exists():
    p_val = run([sys.executable, "training/validate_dataset.py",
                 f"{OUT_PREFIX}.train.jsonl", f"{OUT_PREFIX}.val.jsonl", f"{OUT_PREFIX}.test.jsonl",
                 "--preamble", "prompts/system_preamble_v0.txt"])
    VALIDATE_EXIT = p_val.returncode
else:
    print("Chưa có training/validate_dataset.py trong bản checkout này -> git pull (hoặc chờ PR tương ứng được merge) rồi chạy lại ô này.")
print("validate_dataset.py exit code =", VALIDATE_EXIT)

## Bước 5 — Render MỘT prompt đúng như lúc train / serve

Không lắp tool vào text. Gọi `apply_chat_template(messages[:-1], tools=replay_tools, add_generation_prompt=True)`
của tokenizer Qwen: khối `<tools>…</tools>` phải xuất hiện **trong lượt system**, và target của assistant phải bắt đầu bằng
`<tool_call>`. Bước này chỉ tải tokenizer (~vài MB), không tải model.

In [ ]:
# Render one training prompt exactly as finetune_qlora.py and vLLM do.
PROMPT_HAS_TOOLS = TARGET_IS_TOOL_CALL = None
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
except ImportError:
    tok = None
    print("Chưa có thư viện transformers (Colab có sẵn; local: pip install transformers — bước này không cần torch).")

if tok is not None:
    row = read_jsonl(f"{OUT_PREFIX}.train.jsonl")[0]
    prompt = tok.apply_chat_template(row["messages"][:-1], tools=row["replay_tools"],
                                     tokenize=False, add_generation_prompt=True)
    target = row["messages"][-1]["content"]
    system_turn = prompt.split("<|im_start|>user")[0]
    PROMPT_HAS_TOOLS = "<tools>" in system_turn and "</tools>" in system_turn
    TARGET_IS_TOOL_CALL = target.startswith("<tool_call>")

    print(f"--- prompt (row id={row['id']}) " + "-" * 50)
    print(prompt)
    print("--- assistant target (completion) " + "-" * 45)
    print(target)
    print()
    print(f"'<tools>' xuất hiện trong lượt system : {PROMPT_HAS_TOOLS}")
    print(f"target bắt đầu bằng '<tool_call>'      : {TARGET_IS_TOOL_CALL}")
    assert PROMPT_HAS_TOOLS and TARGET_IS_TOOL_CALL

## Bước 6 — Độ dài token prompt + completion (split train)

F Tuần 1 huấn luyện trên T4 với `--max-len 2560` (bằng `DEFAULT_MAX_LEN` của `training/finetune_qlora.py`); hàng vượt
ngưỡng làm `finetune_qlora.py` **thoát mã 1** (không cắt ngầm), nên nhóm Fine-tune cần biết trước `% > 2560`.
Cũng đếm `% > 4096` (`MAX_SEQ_LENGTH`).

In [ ]:
# Token length = len(prompt tokens) + len(completion + eos tokens), over the TRAIN split.
MAX_LEN_T4, MAX_SEQ_LENGTH = 2560, 4096   # MAX_LEN_T4 == DEFAULT_MAX_LEN in training/finetune_qlora.py
TOKEN_STATS = None
if tok is not None:
    train_rows = read_jsonl(f"{OUT_PREFIX}.train.jsonl")
    lengths = []
    for r in train_rows:
        p = tok.apply_chat_template(r["messages"][:-1], tools=r["replay_tools"], tokenize=False, add_generation_prompt=True)
        c = r["messages"][-1]["content"] + tok.eos_token
        lengths.append(len(tok(p)["input_ids"]) + len(tok(c)["input_ids"]))
    srt = sorted(lengths)
    n = len(srt)
    over_max = sum(l > MAX_LEN_T4 for l in srt)
    over_4096 = sum(l > MAX_SEQ_LENGTH for l in srt)
    TOKEN_STATS = {"n": n, "min": srt[0], "median": srt[n // 2], "p95": srt[int(0.95 * n) - 1], "max": srt[-1],
                   "over_max_len": over_max, "pct_over_max_len": 100 * over_max / n,
                   "over_4096": over_4096, "pct_over_4096": 100 * over_4096 / n}
    for k, v in TOKEN_STATS.items():
        print(f"{k:14s} {v:.2f}" if isinstance(v, float) else f"{k:14s} {v}")

    # Which rows would trip the truncation guard of finetune_qlora.py at --max-len MAX_LEN_T4?
    OVER_ROWS = [(r["id"], l) for r, l in zip(train_rows, lengths) if l > MAX_LEN_T4]
    if OVER_ROWS:
        print(f"\nHàng > {MAX_LEN_T4} token (id: tokens): " + ", ".join(f"{i}: {l}" for i, l in OVER_ROWS))
        print(f"-> finetune_qlora.py --max-len {MAX_LEN_T4} sẽ dừng (exit 1) trên split train này; "
              f"cần --max-len >= {srt[-1]} hoặc loại các hàng trên (báo cho nhóm Fine-tune).")
    else:
        print(f"\nKhông có hàng nào vượt {MAX_LEN_T4} token.")

    try:
        import matplotlib.pyplot as plt
        BLUE, INK, GRID = "#2a78d6", "#52514e", "#e5e5e2"
        fig, ax = plt.subplots(figsize=(7, 3.2))
        ax.hist(lengths, bins=40, color=BLUE, edgecolor="white", linewidth=0.8)
        ax.axvline(MAX_LEN_T4, color=INK, linestyle="--", linewidth=1.5)
        ax.text(MAX_LEN_T4, ax.get_ylim()[1] * 0.9, f" max-len T4 = {MAX_LEN_T4}", color=INK, fontsize=9)
        ax.set_title("Số token prompt + completion (train, 1600 hàng)", loc="left", color=INK)
        ax.set_xlabel("tokens")
        ax.set_ylabel("rows")
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("matplotlib chưa cài -> bỏ qua biểu đồ.")
else:
    print("Bỏ qua (không có tokenizer).")

## Tạo báo cáo

Copy khối in ra bên dưới (từ dòng `## Báo cáo …`) và dán vào **comment** của task trên ClickUp.

In [ ]:
# Print the ClickUp comment block with exactly the numbers the task's acceptance criteria ask for.
import datetime

TASK = "Tuần 2 — Tiền xử lý về định dạng hàng huấn luyện (parity, `<tool_call>`) và chia train/val/test"
lines = [
    f"## Báo cáo {TASK} — {datetime.date.today().isoformat()} — {AUTHOR}",
    f"- Notebook: `notebooks/data/02_preprocess_to_chatml.ipynb` @ `{GIT_SHA}` · môi trường: {'Colab' if IN_COLAB else 'local'}",
    f"- Lệnh: `python datagen/convert_xlam.py --in data/public/xlam_raw_2k.jsonl --out-prefix /tmp/xlam_2k "
    f"--system-file prompts/system_preamble_v0.txt --seed {SEED}` → exit {p_convert.returncode}",
]
for s, d in SPLITS.items():
    lines.append(f"- {s}: {d['rows']} hàng · {d['tool_sets']} tool-set · ids_sha256 `{d['ids_sha256'][:16]}` · "
                 f"split matches committed: **{d['match']}** (bytes identical: {d['bytes_match']})")
lines += [
    f"- eval.json: {len(eval_new)} bản ghi · ids match committed: **{EVAL_MATCH}**",
    f"- tool-set dùng chung giữa các split: **{len(SHARED)}** (yêu cầu 0)",
    f"- `training/validate_dataset.py` exit code: **{VALIDATE_EXIT}** (yêu cầu 0)",
    f"- Prompt render có `<tools>` trong lượt system: **{PROMPT_HAS_TOOLS}** · target bắt đầu bằng `<tool_call>`: **{TARGET_IS_TOOL_CALL}**",
]
if TOKEN_STATS:
    lines.append(f"- Token prompt+completion (train, n={TOKEN_STATS['n']}): min {TOKEN_STATS['min']} · median {TOKEN_STATS['median']} · "
                 f"p95 {TOKEN_STATS['p95']} · max {TOKEN_STATS['max']} · > {MAX_LEN_T4}: {TOKEN_STATS['over_max_len']} hàng "
                 f"(**{TOKEN_STATS['pct_over_max_len']:.2f}%**) · > 4096: {TOKEN_STATS['over_4096']} hàng ({TOKEN_STATS['pct_over_4096']:.2f}%)")
    if OVER_ROWS:
        lines.append(f"- Hàng > {MAX_LEN_T4} token: " + ", ".join(f"`{i}` ({l})" for i, l in OVER_ROWS)
                     + f" → F Tuần 1 cần `--max-len >= {TOKEN_STATS['max']}` hoặc loại các hàng này")
else:
    lines.append("- Token prompt+completion: chưa đo (không có tokenizer)")
print("\n".join(lines))